KÜTÜPHANELER

In [ ]:
! pip install langchain qdrant_client openai tiktoken

QDRANT BAĞLANTI AYARLARI

In [ ]:

from langchain.vectorstores import Qdrant
from langchain.embeddings.openai import OpenAIEmbeddings
from  qdrant_client import models
import qdrant_client
import os

# embeding işlemlerini  açık kaynaklı model ilede yapabiliriz

os.environ['QDRANT_HOST'] = 'QDRANT_HOST'
os.environ['QDRANT_API_KEY'] = 'QDRANT_API_KEY'

client = qdrant_client.QdrantClient(
    os.getenv("QDRANT_HOST"),
    api_key=os.getenv("QDRANT_API_KEY")
)

QDRANT COLLECTION VARLIĞINI SORGULAMA

In [81]:
# Koleksiyonları al
collections = client.get_collections()
collections

CollectionsResponse(collections=[CollectionDescription(name='mng-table'), CollectionDescription(name='star_charts')])

In [ ]:

import os
import json
import openai

# OPENAI API anahtarını ayarlama
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

# JSON dosyasını açıp veriyi okuma
with open('C:/Users/info/OneDrive/Masaüstü/GIT Commit Project/Gemini-ChatBot/d/cleaned_tasks.json', 'r') as file:

    json_data = json.load(file)

# OpenAI API'ye bağlanma
for item in json_data:
    # 'adsoyad', 'title' veya 'description' gibi kolonlardan metni al
    text_input = item.get('adsoyad', '') + " " + item.get('title', '') + " " + item.get('description', '')

    if text_input.strip():  # Eğer metin boş değilse
        # Embedding oluşturma
        response = openai.Embedding.create(
            input=text_input,
            model="text-embedding-3-small"
        )

        # Embedding sonucunu al
        embedding = response['data'][0]['embedding']

        # Sonuçları yazdırma veya kaydetme
        print(f"ID: {item.get('id', '')} - AdSoyad: {item.get('adsoyad', '')} - Embedding: {embedding}")


QDRANT COLLECTION OLUŞTURMA

In [84]:
from qdrant_client.models import VectorParams

client.create_collection(
    collection_name="test-2",
    vectors_config=VectorParams(
        size=768,  # Vector size (embedding dimension)
        distance="Cosine"  # Use cosine similarity for distance metric
    )
)

True

In [87]:
# ! pip install fastembed

In [ ]:
#! pip install transformers -U

QDRANT BAĞLANTI AYARLARI

In [77]:
# create collections

import qdrant_client.http


os.environ["QDRANT_COLLECTION_NAME"] = "star_charts"
vectors_config = qdrant_client.http.models.VectorParams(
    size=1536,
    distance=models.Distance.COSINE
    )

client.recreate_collection(
    collection_name= os.getenv("QDRANT_COLLECTION_NAME"),
    vectors_config= vectors_config
)


C:\Users\info\AppData\Local\Temp\ipykernel_21148\3868407671.py:12: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

QDRANT EMBEDDING İŞLEMİ

In [ ]:
# create vector store
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"
embeddings = OpenAIEmbeddings()
vector_store = Qdrant(
    client=client,
    collection_name=os.getenv("QDRANT_COLLECTION_NAME"),
    embeddings=embeddings,
)



Veri Setleri Düzenleme

Mng Veri Seti

In [72]:
import json 
from langchain.text_splitter import CharacterTextSplitter
import os
# JSON dosyasındaki tüm alanları alıp embedding için işle
def process_json_for_embedding(file_path):
    with open(file_path, encoding='utf-8') as f:
        tur_veri = json.load(f)

    # JSON verisini işleyerek tüm alanları birleştiriyoruz
    for item in tur_veri:
        # JSON öğesinin tüm metin alanlarını birleştiriyoruz
        fields = [
            item.get('id', ''),
            item.get('title', ''),
            item.get('description', ''),
            item.get('adsoyad', '')
        ]
        text = ' '.join(fields)

        # Metin parçalama işlemi
        chunks = get_chunks(text)

        # Her bir metin parçasını embedding yapıp Qdrant'a ekliyoruz
        vector_store.add_texts(chunks)
        print(f"Veri işlendi: {item['id']} - {item['title']}")

# Metinleri parçalara ayırmak için fonksiyon
def get_chunks(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    return chunks

# JSON dosyasının yolu
directory_path = "C:/Users/info/OneDrive/Masaüstü/GIT Commit Project/Gemini-ChatBot/d/" 

# Klasör yolunu buraya yazın
for filename in os.listdir(directory_path):
    if filename.endswith(".json"):
        file_path = os.path.join(directory_path, filename)
        process_json_for_embedding(file_path)

Veri işlendi: 2508 - A2NUser - Login ekranı ve dashboard
Veri işlendi: 2514 - A2N Router için alt yapı kurulumu
Veri işlendi: 2515 - A2NRouter - Routemap fonksiyonları
Veri işlendi: 2516 - V2Admin - Acente detayı - A2 Network sekmesi
Veri işlendi: 2517 - V2admin - A2 network - Route Map
Veri işlendi: 2518 - V2Admin - A2 Network - Route Test
Veri işlendi: 2519 - A2 - A2N - AccountDetails
Veri işlendi: 2520 - A2NRouter - Relay fonksiyonu
Veri işlendi: 2522 - A2 - A2N - ListCategories
Veri işlendi: 2523 - A2 - A2N - ListProducts
Veri işlendi: 2525 - A2 - A2N - ProductDetails
Veri işlendi: 2526 - A2 - A2N - Otel ve tur sqllerinde değişiklik
Veri işlendi: 2529 - A2NRouter - relay datasının taşınması
Veri işlendi: 2539 - A2 - A2N - CreateAccount
Veri işlendi: 2547 - A2 - A2NMaster - MasterCreateAccount
Veri işlendi: 2548 - A2 - A2N - UpdateA2NID
Veri işlendi: 2549 - A2 - A2NMaster - Kullanıcı ekleme ve MasterAddUser
Veri işlendi: 2550 - A2 - A2NMaster - MasterUpdatePassword
Veri işlendi: 255

JSON DOSYALARINI VEKTÖREL OLARAK QDRANT İÇİNE EKLE

In [79]:
# plug vector store into  retrieval chain
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI

qa =  RetrievalQA.from_chain_type(
    llm = OpenAI(),
    chain_type = "stuff",
    retriever = vector_store.as_retriever()
)

QDRANT TEST

In [ ]:
# plug vector store into  retrieval chain
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI


# create collections

import qdrant_client.http


os.environ["QDRANT_COLLECTION_NAME"] = "star_charts"

vectors_config = qdrant_client.http.models.VectorParams(
    size=1536,
    distance=models.Distance.COSINE
    )

client.recreate_collection(
    collection_name= os.getenv("QDRANT_COLLECTION_NAME"),
    vectors_config= vectors_config
)

# create vector store
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"
embeddings = OpenAIEmbeddings()
vector_store = Qdrant(
    client=client,
    collection_name=os.getenv("QDRANT_COLLECTION_NAME"),
    embeddings=embeddings,
)

qa =  RetrievalQA.from_chain_type(
    llm = OpenAI(),
    chain_type = "stuff",
    retriever = vector_store.as_retriever()
)

query = "projedeki ilk task nedir"
response = qa.run(query)
print(response)

C:\Users\info\AppData\Local\Temp\ipykernel_21148\703320756.py:18: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


 I'm sorry, I don't have enough context to answer that question. Please provide more information about the project and the tasks involved.


QDRANT COLLECTION GÖRME

In [57]:
# if you want to delete vector store

# Tüm koleksiyonları alıyoruz
collections = client.get_collections()

collections

CollectionsResponse(collections=[CollectionDescription(name='testt')])

QDRANT COLLECTION SİLME

In [56]:
collection_name = os.getenv("QDRANT_COLLECTION_NAME")

client.delete_collection(collection_name=collection_name)


True

In [ ]:
client 

In [1]:
# import os
# import json

# # JSON dosyalarının bulunduğu dizin
# dizin = "C:/Users/info/OneDrive/Masaüstü/GIT Commit Project/Gemini-ChatBot/clean data/data/"  # Dizininize göre güncelleyin

# # Dizin içindeki tüm JSON dosyalarını oku
# for dosya_adi in os.listdir(dizin):
#     if dosya_adi.endswith('.json'):
#         dosya_yolu = os.path.join(dizin, dosya_adi)
        
#         # JSON dosyasını oku
#         with open(dosya_yolu, 'r', encoding='utf-8') as dosya:
#             veri = json.load(dosya)
        
#         # 'kalkissehir' ve 'vizeulke' alanlarını kaldır
#         for tur in veri:
#             tur.pop('kalkissehir', None)
#             tur.pop('vizeulke', None)
        
#         # Güncellenmiş veriyi tekrar JSON dosyasına yaz
#         with open(dosya_yolu, 'w', encoding='utf-8') as dosya:
#             json.dump(veri, dosya, ensure_ascii=False, indent=4)

# print("Kolonlar başarıyla kaldırıldı.")


Kolonlar başarıyla kaldırıldı.


In [2]:
# import os
# import json

# # JSON dosyalarının bulunduğu dizin
# dizin = "C:/Users/info/OneDrive/Masaüstü/GIT Commit Project/Gemini-ChatBot/clean data/data/"  # Dizininize göre güncelleyin

# # Dizin içindeki tüm JSON dosyalarını oku
# for dosya_adi in os.listdir(dizin):
#     if dosya_adi.endswith('.json'):
#         dosya_yolu = os.path.join(dizin, dosya_adi)
        
#         # JSON dosyasını oku
#         with open(dosya_yolu, 'r', encoding='utf-8') as dosya:
#             veri = json.load(dosya)
        
#         # 'kalkissehir' ve 'vizeulke' alanlarını kaldır, 'kesinkalkis' alanını güncelle
#         for tur in veri:
#             if tur.get('kesinkalkis') == "0":
#                 tur['kesinkalkis'] = "Bu tur kesin kalkışlı değildir"
        
#         # Güncellenmiş veriyi tekrar JSON dosyasına yaz
#         with open(dosya_yolu, 'w', encoding='utf-8') as dosya:
#             json.dump(veri, dosya, ensure_ascii=False, indent=4)

# print("Kolonlar kaldırıldı ve 'kesinkalkis' alanı güncellendi.")


Kolonlar kaldırıldı ve 'kesinkalkis' alanı güncellendi.


In [ ]:
import json
import os

from qdrant_client import QdrantClient
from qdrant_client.http import models
from langchain.embeddings import OpenAIEmbeddings



# Qdrant ve OpenAI API anahtarlarını ayarlama
os.environ['QDRANT_HOST'] = 'QDRANT_HOST'
os.environ['QDRANT_API_KEY'] = 'QDRANT_API_KEY'

client = QdrantClient(
    os.getenv("QDRANT_HOST"),
    api_key=os.getenv("QDRANT_API_KEY")
)

os.environ["QDRANT_COLLECTION_NAME"] = "star_charts"

# OpenAI API Anahtarı
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
embeddings_model = OpenAIEmbeddings()

# Qdrant Koleksiyonunu Yeniden Oluştur (Sil & Yarat)
client.recreate_collection(
    collection_name=os.getenv("QDRANT_COLLECTION_NAME"),
    vectors_config=models.VectorParams(
        size=1536,  
        distance=models.Distance.COSINE
    )
)

# JSON Dosya Yolu
file_path = "C:/Users/info/OneDrive/Masaüstü/GIT Commit Project/Gemini-ChatBot/d/cleaned_tasks.json"

# JSON Dosyasını Oku ve Verileri İşle
with open(file_path, encoding="utf-8") as f:
    data = json.load(f)

# Qdrant İçin Veri Hazırlama
documents = []
metadatas = []
ids = []

for index, item in enumerate(data):
    combined_text = f"{item.get('adsoyad', '')} {item.get('title', '')} {item.get('description', '')}"
    documents.append(combined_text)
    metadatas.append({
        "adsoyad": item.get("adsoyad", ""),
        "title": item.get("title", ""),
        "description": item.get("description", ""),
        "text": combined_text  # Ham metni de ekle
    })
    ids.append(index)

# Embedding Hesapla (Her 10 Veri İçin)
batch_size = 10
for i in range(0, len(documents), batch_size):
    batch_texts = documents[i:i + batch_size]
    batch_embeddings = embeddings_model.embed_documents(batch_texts)

    # Qdrant'a Ekle
    client.upsert(
        collection_name=os.getenv("QDRANT_COLLECTION_NAME"),
        points=[
            models.PointStruct(
                id=ids[i + j],
                vector=batch_embeddings[j],
                payload=metadatas[i + j]  # Ham veri burada saklanıyor
            )
            for j in range(len(batch_embeddings))
        ]
    )

    print(f"{i + batch_size} vektör işlendi ve Qdrant'a eklendi.")


In [ ]:
import os
import json
from bs4 import BeautifulSoup

# HTML içeriğini temizleyen fonksiyon
def clean_html(html_text):
    # BeautifulSoup ile HTML taglerini kaldırıyoruz
    soup = BeautifulSoup(html_text, "html.parser")
    clean_text = soup.get_text(separator=" ", strip=True)  # Tagler dışında sadece metin kalacak

    # \r\n (carriage return ve line feed) karakterlerini temizle
    clean_text = clean_text.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ')
    
    return clean_text

# JSON dosyasındaki tüm alanları alıp embedding için işle
def process_json_for_embedding(file_path, output_path):
    with open(file_path, encoding='utf-8') as f:
        tur_veri = json.load(f)

    # JSON verisini işleyerek tüm alanları birleştiriyoruz
    cleaned_data = []
    for item in tur_veri:
        # Temizlenmiş description (HTML tagleri ve \r\n karakterleri kaldırılmış)
        cleaned_description = clean_html(item.get('description', ''))

        # Yeni formatta düzenlenmiş veriyi oluştur
        cleaned_data.append({
            "id": item.get('id', ''),
            "title": item.get('title', ''),
            "description": cleaned_description,
            "adsoyad": item.get('adsoyad', '')
        })

    # İşlenmiş veriyi yeni dosyaya kaydediyoruz
    with open(output_path, 'w', encoding='utf-8') as output_file:
        json.dump(cleaned_data, output_file, ensure_ascii=False, indent=2)

    print(f"Veri {output_path} dosyasına kaydedildi.")
    return cleaned_data


# JSON dosyasının yolu
directory_path = "C:/Users/info/OneDrive/Masaüstü/GIT Commit Project/Gemini-ChatBot/clean data/"  # Klasör yolunu buraya yazın
output_directory = "C:/Users/info/OneDrive/Masaüstü/GIT Commit Project/Gemini-ChatBot/clean data/"  # Çıktı dosyasının yolu

# Dosyaları işle ve çıktı olarak kaydet
for filename in os.listdir(directory_path):
    if filename.endswith(".json"):
        file_path = os.path.join(directory_path, filename)
        output_file_path = os.path.join(output_directory, f"cleaned_{filename}")
        process_json_for_embedding(file_path, output_file_path)
